# GNN — 1 Lepton 2 Taus (Run 2)

Graph-neural-network counterpart to `DNN.ipynb`'s flat-feature MLP, on the
same 1l2tau Run 2 data (`Evie/PPSSP_2026/1l2tau/run2`), same preselection,
same leakage-free feature policy, and the same deterministic 80/10/10
train/val/test split — so results are directly comparable.

**Representation change.** The MLP concatenates every event into one flat
feature vector (kinematics + hand-engineered pairwise variables like `dR_*`,
`m_*`, `HT_*`). This notebook instead represents each event as a small graph:
one node per reconstructed physics object (lepton, tau1, tau2, jet1, jet2,
MET), each carrying only its own kinematics, with a fully-connected edge set
within the event. The hand-engineered pairwise/aggregate variables are
deliberately **not** given to the model — the point of the GNN is to see how
much of that relational information (angular separations, invariant masses,
combined momenta) message passing can recover on its own from raw 4-vectors.

**Scope.** This mirrors the core of the MLP pipeline — data loading, graph
construction, model, training with early stopping, ROC/AUC evaluation, and a
single held-out test evaluation — for Run 2 only. Permutation importance and
correlation pruning are feature-level diagnostics that don't map cleanly onto
graph nodes, so they're intentionally left out here (see `DNN.ipynb` for
those). Run 3 / Combined tracks and the hyperparameter grid search are also
left out to keep this a focused first pass; the structure below is written so
they're straightforward to add later, the same way `DNN.ipynb` does.

## Libraries

In [ ]:
import os

# Must be set BEFORE CUDA/cuBLAS initializes for deterministic cuBLAS matmul.
# If this kernel already has CUDA initialized (e.g. you've run cells before
# adding this), these two env vars won't take effect until you RESTART THE
# KERNEL.

os.environ["PYTHONHASHSEED"] = "42"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# torch and uproot (via numpy) can each bundle their own OpenMP runtime;
# loading both in one process aborts the kernel on some platforms
# ("OMP: Error #15 ... libomp.dylib already initialized") unless this is
# set before either is imported.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import uproot
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as GeoDataLoader
from torch_geometric.nn import GATv2Conv, global_mean_pool, global_max_pool

RANDOM_STATE = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int = RANDOM_STATE):

    """
    Full determinism, not just seeding: also pins cuDNN to deterministic
    kernels, disables its autotuner, and asks torch to error out (rather than
    silently fall back) on any op without a deterministic implementation.
    Same convention as DNN.ipynb's set_seed, so re-seeded runs here are
    reproducible the same way. Determinism is only guaranteed on the SAME
    machine / CUDA / torch version - it is not portable across hardware or
    library versions.
    """

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)


set_seed(RANDOM_STATE)

print(f"Using device: {DEVICE}")

In [ ]:
# ---------------------------------------------------------------------------
# Paths - 1 lepton + 2 taus, Run 2 (same source files as DNN.ipynb / the
# Final_Notebooks/1L2Tau_Master_Pipeline.ipynb, so results are comparable).
# ---------------------------------------------------------------------------

BASE_DIR_RUN2 = Path("PPSSP_2026/1l2tau/run2")
BASE_DIR = BASE_DIR_RUN2
ACTIVE_RUN = "Run 2"
TREE_NAME = "AnalysisMiniTree"

# ---------------------------------------------------------------------------
# Preselection (see repo README.md, "1 Lepton 2 Taus")
# ---------------------------------------------------------------------------

PRESELECTION = "(n_b_jet == 0) & (n_jet >= 2)"

# ---------------------------------------------------------------------------
# Processes: filename + label (1 = signal, 0 = background)
# ---------------------------------------------------------------------------

FILES = {
    "signal_ggF": ("signal_ggF.root", 1),
    "signal_VBF": ("signal_VBF.root", 1),
    "Diboson":    ("diboson.root",    0),
    "Zjets":      ("Zjets.root",      0),
    "Wjets":      ("Wjets.root",      0),
    "ttbar":      ("ttbar.root",      0),
    "tops":       ("tops.root",       0),
    "SingleH":    ("singleH.root",    0),
    "Vgamma":     ("Vgamma.root",     0),
    "VVV":        ("VVV.root",        0),
}

WEIGHT_PARTS = ["weight", "weights"]  # raw branches; w_phys = their product

# ---------------------------------------------------------------------------
# Leakage-free feature-selection policy - IDENTICAL to DNN.ipynb / the
# XGBoost master pipeline, so this notebook builds graph nodes only out of
# branches the other models are also allowed to see.
# ---------------------------------------------------------------------------

BLOCK_SUBSTR = ["weight", "effsf", "_ff", "truth", "istrue", "fake", "anti",
                "dsid", "eventnumber", "_RNNTight", "_isOS", "_d0sig"]
BLOCK_EXACT = {
    "n_b_jet", "pass1l2tau", "hhml_subchannelflavor",
    "tau1_RNNJetScoreSigTrans", "tau2_RNNJetScoreSigTrans",
    "pair_isOStaus", "pair_isOSleptau", "tau2_baseline_RNNTight", "l1_d0sig", "tau1_charge", "tau2_charge", "mZ_veto", "tau1_decayMode", "tau2_decayMode", "tau1_nprong", "tau2_nprong"
}

BLOCK_EXACT_LOWER = {b.lower() for b in BLOCK_EXACT}


def is_feature(branch: str) -> bool:

    """True if `branch` is safe to use as a training feature (see policy above)."""

    lb = branch.lower()
    return lb not in BLOCK_EXACT_LOWER and not any(s.lower() in lb for s in BLOCK_SUBSTR)

## Data Loading Helpers

Identical to `DNN.ipynb`'s loaders - same discovery, same preselection, same
sentinel-to-NaN cleaning - so `data`/`features` here are byte-identical to
the MLP notebook's Run 2 `data_run2`/`features_run2`.

In [ ]:
def discover_common_features(base_dir, files=FILES, tree_name=TREE_NAME):

    """
    Branches common to EVERY process file in `base_dir`, filtered through`is_feature`.
    """

    common = None

    for fname, _ in files.values():
        keys = set(uproot.open({str(Path(base_dir) / fname): tree_name}).keys())
        common = keys if common is None else common & keys

    features = sorted(b for b in common if is_feature(b))

    print(f"{len(features)} candidate features (common to all {len(files)} processes, leakage-free)")

    return features


def load_run_data(base_dir, features, files=FILES, weight_parts=WEIGHT_PARTS,
                   preselection=PRESELECTION, tree_name=TREE_NAME, verbose=True):

    """
    Read every process file under `base_dir`, apply the preselection at
    read time, and concatenate into one DataFrame with bookkeeping columns:
      - w_phys  : physical event weight = weight * weights
      - label   : 1 = signal, 0 = background
      - process : originating process name
    """

    base_dir = Path(base_dir)
    dfs = []

    for proc, (fname, label) in files.items():

        tree = uproot.open({str(base_dir / fname): tree_name})
        df = tree.arrays(features + weight_parts, cut=preselection, library="pd").head(4000)
        df["w_phys"] =  df["weights"] * df["weight"]
        df["label"] = label
        df["process"] = proc
        dfs.append(df)

        if verbose:
            print(f"{proc:12s}: {len(df):>8d} events after preselection")

    return pd.concat(dfs, ignore_index=True)


def clean_data(data, features, verbose=True):

    """
    Post-concat cleaning: drop constant/empty features, then mask sentinel
    values (< -100, e.g. -999) to NaN. Returns (cleaned_data, updated_features);
    operates on a copy, does not mutate the input DataFrame.
    """

    data = data.copy()
    nun = data[features].nunique()
    const = nun[nun <= 1].index.tolist()
    features = [f for f in features if f not in const]
    data = data.drop(columns=const)

    if verbose:
        print(f"Dropped {len(const)} constant/empty features:\n  {sorted(const)}")

    for f in features:
        m = data[f] < -100
        if m.any():
            if verbose:
                print(f"  sentinel -> NaN: {f} ({m.mean():.1%})")
            data[f] = data[f].mask(m)

    if verbose:
        print(f"\n{len(features)} final features")
        print(f"Total: {len(data)} events | signal = {(data.label==1).sum()} | "
              f"background = {(data.label==0).sum()}")
        print(f"Yield (w_phys): signal = {data.loc[data.label==1,'w_phys'].sum():.2f} | "
              f"background = {data.loc[data.label==0,'w_phys'].sum():.2f}")
    return data, features

## Load Run 2 Data

In [ ]:
candidate_features = discover_common_features(BASE_DIR_RUN2)
data = load_run_data(BASE_DIR_RUN2, candidate_features)
data, features = clean_data(data, candidate_features)

print(f"\nLoaded {len(data)} events | {len(features)} leakage-free features available")

## Sentinel Audit (-1)

Same diagnostic as `DNN.ipynb`: `clean_data` only masks `< -100` sentinels
(e.g. `-999`) to NaN. Some ATLAS branches instead default to exactly `-1`
for "undefined", which is also a perfectly valid value for other branches
(e.g. charges) - so it must **not** be blanket-masked. Kept empty here to
match the MLP notebook's audited conclusion (no genuine `-1` sentinels among
the leakage-free features for this dataset); re-run and inspect
`neg1_df` if the upstream ntuples change.

In [ ]:
NEG1_SENTINEL_FEATURES = set()

neg1_rows = []
for f in features:
    vals = data[f]
    frac_neg1 = (vals == -1).mean()
    if frac_neg1 == 0:
        continue
    above = vals[vals > -1]
    gap = (above.min() - (-1)) if len(above) else np.nan
    neg1_rows.append({"feature": f, "frac_exactly_-1": frac_neg1, "gap_to_next_value_above": gap})

neg1_df = pd.DataFrame(neg1_rows).sort_values("frac_exactly_-1", ascending=False)
print(f"{len(neg1_df)} / {len(features)} features have at least one row exactly equal to -1:")
print(neg1_df.to_string(index=False))

for f in NEG1_SENTINEL_FEATURES:
    data[f] = data[f].mask(data[f] == -1)

if NEG1_SENTINEL_FEATURES:
    print(f"\nMasked -1 -> NaN for: {sorted(NEG1_SENTINEL_FEATURES)}")
else:
    print("\nNEG1_SENTINEL_FEATURES is empty - no -1 values masked.")

## Train/Validation/Test Split

Same deterministic two-stage stratified **80/10/10** split (`make_3way_split`)
as `DNN.ipynb`, on the same `data` - so TEST here is the same set of events as
the MLP notebook's TEST, cross-checked against its persisted partition.
TEST is held out from everything below (imputation, scaling, early stopping)
until the single "Held-Out Test Evaluation" cell at the end.

In [ ]:
def make_fit_weights(labels, abs_weights):

    """
    Balance signal/background total weight and normalize the mean weight
    to 1. `abs_weights` must already be non-negative (Sherpa weights can be
    negative).
    """

    labels = np.asarray(labels)

    fit_weights = np.asarray(abs_weights, dtype=float).copy()
    sum_signal = fit_weights[labels == 1].sum()
    sum_background = fit_weights[labels == 0].sum()
    fit_weights[labels == 1] *= sum_background / sum_signal
    fit_weights *= len(fit_weights) / fit_weights.sum()

    return fit_weights


def make_3way_split(data, test_size=0.10, val_size=0.10, seed=RANDOM_STATE, extra_stratify_col=None):

    """
    Deterministic two-stage stratified 80/10/10 train/val/test split -
    IDENTICAL logic (same seed, same stratify) to DNN.ipynb's
    make_3way_split, so both notebooks land on the same partitions whenever
    `data` is built identically (same FILES order -> concat -> clean_data).
    """

    def _strata(df):
        if extra_stratify_col is not None:
            return df["label"].astype(str) + "_" + df[extra_stratify_col].astype(str)
        return df["label"]

    trainval_df, test_df = train_test_split(data, test_size=test_size, random_state=seed, stratify=_strata(data))

    val_frac_of_remaining = val_size / (1 - test_size)
    train_df, val_df = train_test_split(
        trainval_df, test_size=val_frac_of_remaining, random_state=seed, stratify=_strata(trainval_df)
    )

    for name, df in (("Train", train_df), ("Val", val_df), ("Test", test_df)):
        sig_n, bkg_n = int((df.label == 1).sum()), int((df.label == 0).sum())
        sig_y = df.loc[df.label == 1, "w_phys"].sum()
        bkg_y = df.loc[df.label == 0, "w_phys"].sum()
        print(f"{name:5s}: {len(df):>8d} events | signal = {sig_n:>7d} (yield={sig_y:>10.2f}) | "
              f"background = {bkg_n:>7d} (yield={bkg_y:>10.2f}) | "
              f"signal weight scale factor = {bkg_y / sig_y:.1f}")

    return train_df, val_df, test_df


def assert_same_test_partition(test_df, path):

    """
    Cross-notebook guard: if a test partition already exists on disk at
    `path` (written by DNN.ipynb / the master pipeline reading this same
    BASE_DIR), verify this notebook's freshly-computed `test_df` is
    equal to it on their shared columns.
    """

    path = Path(path)
    if not path.exists():
        print(f"  (no prior test partition at {path} yet - nothing to cross-check)")
        return

    prior_df = uproot.open({str(path): "tree"}).arrays(library="pd")
    cols = sorted((set(test_df.columns) & set(prior_df.columns)) - {"run"})
    a = test_df[cols].sort_values(cols).reset_index(drop=True)
    b = prior_df[cols].sort_values(cols).reset_index(drop=True)

    pd.testing.assert_frame_equal(a, b, check_exact=False, rtol=1e-5, atol=1e-8, check_dtype=False)
    print(f"  cross-check OK: {path} test partition matches on {len(cols)} shared columns ({len(test_df)} rows)")


train_df, val_df, test_df = make_3way_split(data)

SPLIT_DIR_RUN2 = BASE_DIR_RUN2 / "splits"
assert_same_test_partition(test_df, SPLIT_DIR_RUN2 / "test.root")

## Object / Node Schema

Six nodes per event, one per reconstructed object, using only branches that
survive the leakage-free `is_feature` policy above and that `clean_data`
didn't drop as constant (checked against this dataset: the tau `_base_id` /
`_passOR` / `_besline_RNNMedium_eleid` flags are constant post-preselection
and so aren't available - only kinematics remain for tau1/tau2).

| node   | source prefix | fields                          |
|--------|---------------|----------------------------------|
| lepton | `l1_`         | pt, eta, phi, e, charge, pdg (-> is_electron) |
| tau1   | `tau1_`       | pt, eta, phi                     |
| tau2   | `tau2_`       | pt, eta, phi                     |
| jet1   | `j1_`         | pt, eta, phi, e                  |
| jet2   | `j2_`         | pt, eta, phi, e                  |
| MET    | `met_`        | met (-> pt slot), phi, sumet (-> e slot) |

Each node's feature vector is `[pt, eta, sin(phi), cos(phi), e, charge,
is_electron, is_lepton, is_tau, is_jet, is_met]` (11 dims) - fields that
don't apply to an object (e.g. tau charge, blocked as leakage; MET eta,
undefined) are zero-filled, and the 4-way type one-hot lets the (shared)
graph-conv weights tell object roles apart. `phi` is encoded as
`(sin, cos)` rather than a raw scaled scalar so its circular topology
(`-pi` and `+pi` are the same angle) isn't broken by standard scaling.

In [ ]:
OBJECT_COLUMNS = {
    "lepton": {"pt": "l1_pt",   "eta": "l1_eta",   "phi": "l1_phi",   "e": "l1_e", "charge": "l1_charge", "pdg": "l1_pdg"},
    "tau1":   {"pt": "tau1_pt", "eta": "tau1_eta", "phi": "tau1_phi"},
    "tau2":   {"pt": "tau2_pt", "eta": "tau2_eta", "phi": "tau2_phi"},
    "jet1":   {"pt": "j1_pt",   "eta": "j1_eta",   "phi": "j1_phi",   "e": "j1_e"},
    "jet2":   {"pt": "j2_pt",   "eta": "j2_eta",   "phi": "j2_phi",   "e": "j2_e"},
    "met":    {"pt": "met_met", "phi": "met_phi",  "e": "met_sumet"},
}
NODE_ORDER = ["lepton", "tau1", "tau2", "jet1", "jet2", "met"]
NODE_TYPE = {"lepton": "lepton", "tau1": "tau", "tau2": "tau", "jet1": "jet", "jet2": "jet", "met": "met"}
TYPE_LIST = ["lepton", "tau", "jet", "met"]
N_NODES = len(NODE_ORDER)
N_NODE_FEATURES = 7 + len(TYPE_LIST)  # pt, eta, sin(phi), cos(phi), e, charge, is_electron + type one-hot

REQUIRED_OBJECT_COLUMNS = sorted({c for cols in OBJECT_COLUMNS.values() for c in cols.values()})
missing = [c for c in REQUIRED_OBJECT_COLUMNS if c not in features]
assert not missing, f"Node schema references columns dropped by clean_data / the leakage policy: {missing}"

# Continuous columns (imputed + standard-scaled, fit on TRAIN only). phi is
# handled separately via sin/cos; charge and pdg-derived is_electron are
# left unscaled (0/1-ish flags - scaling would just relabel them).
CONTINUOUS_NODE_COLS = sorted({
    c for cols in OBJECT_COLUMNS.values() for k, c in cols.items() if k in ("pt", "eta", "e")
})

print(f"{N_NODES} nodes/event x {N_NODE_FEATURES} features/node")
print(f"{len(CONTINUOUS_NODE_COLS)} continuous columns to impute + scale: {CONTINUOUS_NODE_COLS}")

# Fully-connected edge set - identical topology for every event (n_jet >= 2
# in the preselection guarantees all 6 slots are always populated), so it's
# built once and shared across every graph.
EDGE_INDEX = torch.tensor(
    [[i, j] for i in range(N_NODES) for j in range(N_NODES) if i != j],
    dtype=torch.long,
).t().contiguous()

# ---- Median imputation (fit on TRAIN only), then standard scaling on the
# continuous columns only (fit on TRAIN only). Mirrors DNN.ipynb's
# imputation/scaling convention, restricted to the columns the graph nodes
# actually use.
train_medians = train_df[REQUIRED_OBJECT_COLUMNS].median()

train_imp = train_df[REQUIRED_OBJECT_COLUMNS].fillna(train_medians)
val_imp = val_df[REQUIRED_OBJECT_COLUMNS].fillna(train_medians)
test_imp = test_df[REQUIRED_OBJECT_COLUMNS].fillna(train_medians)

scaler = StandardScaler()
train_scaled = train_imp.copy()
val_scaled = val_imp.copy()
test_scaled = test_imp.copy()
train_scaled[CONTINUOUS_NODE_COLS] = scaler.fit_transform(train_imp[CONTINUOUS_NODE_COLS])
val_scaled[CONTINUOUS_NODE_COLS] = scaler.transform(val_imp[CONTINUOUS_NODE_COLS])
test_scaled[CONTINUOUS_NODE_COLS] = scaler.transform(test_imp[CONTINUOUS_NODE_COLS])

assert np.isfinite(train_scaled.to_numpy()).all(), "NaN/inf reached the model input (train)"
assert np.isfinite(val_scaled.to_numpy()).all(), "NaN/inf reached the model input (val)"
assert np.isfinite(test_scaled.to_numpy()).all(), "NaN/inf reached the model input (test)"

print("Imputation + scaling done (fit on train only).")

## Graph Construction

Vectorized build of the `(n_events, N_NODES, N_NODE_FEATURES)` node-feature
tensor for each split, then wrapped one `torch_geometric.data.Data` object
per event (all sharing the same precomputed `EDGE_INDEX`).

In [ ]:
def stack_node_features(df_scaled, df_imp):

    """
    Vectorized construction of the (n_events, N_NODES, N_NODE_FEATURES) node
    tensor. `df_scaled` supplies pt/eta/e (imputed + standard-scaled);
    `df_imp` supplies phi/charge/pdg (imputed but NOT scaled - phi goes
    through sin/cos, charge/pdg are categorical-ish and left raw).
    """

    n = len(df_scaled)
    node_arrays = []

    for name in NODE_ORDER:
        cols = OBJECT_COLUMNS[name]

        pt = df_scaled[cols["pt"]].to_numpy(dtype=np.float32)
        eta = df_scaled[cols["eta"]].to_numpy(dtype=np.float32) if "eta" in cols else np.zeros(n, dtype=np.float32)
        phi = df_imp[cols["phi"]].to_numpy(dtype=np.float32)
        e = df_scaled[cols["e"]].to_numpy(dtype=np.float32) if "e" in cols else np.zeros(n, dtype=np.float32)
        charge = df_imp[cols["charge"]].to_numpy(dtype=np.float32) if "charge" in cols else np.zeros(n, dtype=np.float32)
        is_electron = (
            (np.abs(df_imp[cols["pdg"]].to_numpy()) == 11).astype(np.float32)
            if "pdg" in cols else np.zeros(n, dtype=np.float32)
        )
        type_onehot = np.tile(
            np.array([float(NODE_TYPE[name] == t) for t in TYPE_LIST], dtype=np.float32), (n, 1)
        )

        node_arrays.append(np.column_stack(
            [pt, eta, np.sin(phi), np.cos(phi), e, charge, is_electron, type_onehot]
        ))

    return np.stack(node_arrays, axis=1)  # (n, N_NODES, N_NODE_FEATURES)


def build_graph_dataset(df_scaled, df_imp, df_raw, fit_weights=None):

    """
    Build a list of Data(x, edge_index, y, w[, w_eval]) objects, one per
    event in `df_raw` (which supplies label/w_phys). If `fit_weights` is
    given (class-balanced + mean-normalized training weights), `w` holds
    those and `w_eval` holds plain |w_phys| for eval-mode metrics -
    otherwise `w` alone holds |w_phys| (the val/test convention).
    """

    node_features = torch.from_numpy(stack_node_features(df_scaled, df_imp))
    y = torch.from_numpy(df_raw["label"].to_numpy(dtype=np.float32))
    w_abs = torch.from_numpy(np.abs(df_raw["w_phys"].to_numpy(dtype=np.float32)))
    w_fit = torch.from_numpy(fit_weights.astype(np.float32)) if fit_weights is not None else None

    dataset = []
    for i in range(node_features.shape[0]):
        kwargs = dict(x=node_features[i], edge_index=EDGE_INDEX, y=y[i:i + 1])
        if w_fit is not None:
            kwargs["w"] = w_fit[i:i + 1]
            kwargs["w_eval"] = w_abs[i:i + 1]
        else:
            kwargs["w"] = w_abs[i:i + 1]
        dataset.append(Data(**kwargs))
    return dataset


w_train_fit = make_fit_weights(train_df["label"].to_numpy(), np.abs(train_df["w_phys"].to_numpy()))

train_dataset = build_graph_dataset(train_scaled, train_imp, train_df, fit_weights=w_train_fit)
val_dataset = build_graph_dataset(val_scaled, val_imp, val_df)
test_dataset = build_graph_dataset(test_scaled, test_imp, test_df)

print(f"Built {len(train_dataset)} train / {len(val_dataset)} val / {len(test_dataset)} test graphs")
print(f"Example graph: {train_dataset[0]}")

BATCH_SIZE = 2048
train_loader_fit = GeoDataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
train_loader_eval = GeoDataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
val_loader = GeoDataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = GeoDataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## GNN Model

Two `GATv2Conv` message-passing layers (attention over the fully-connected
neighborhood learns which object pairs matter per event, the relational
analogue of the MLP's hand-picked `dR_*`/`m_*` features), then
mean+max graph-level pooling, then a small MLP head - same dropout +
single-logit-output convention as `DNN.ipynb`'s `SimpleMLP`.

In [ ]:
DEFAULT_HIDDEN_CHANNELS = 64
DEFAULT_DROPOUT = 0.3


class ObjectGNN(nn.Module):

    """
    Object-level graph classifier: 2x GATv2Conv, mean+max pooling, MLP head,
    single output logit (paired with BCEWithLogitsLoss for numerical
    stability, same convention as DNN.ipynb's SimpleMLP).
    """

    def __init__(self, in_channels, hidden_channels=DEFAULT_HIDDEN_CHANNELS, dropout=DEFAULT_DROPOUT):

        super().__init__()
        self.conv1 = GATv2Conv(in_channels, hidden_channels)
        self.conv2 = GATv2Conv(hidden_channels, hidden_channels)
        self.dropout = dropout
        self.head = nn.Sequential(
            nn.Linear(hidden_channels * 2, hidden_channels), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_channels, hidden_channels // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_channels // 2, 1),
        )

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch)], dim=1)
        return self.head(x).squeeze(-1)


def build_model(hidden_channels=DEFAULT_HIDDEN_CHANNELS, dropout=DEFAULT_DROPOUT, in_channels=None):

    """Factory so training/hyperparameter code can build a fresh model per trial."""

    if in_channels is None:
        in_channels = N_NODE_FEATURES
    return ObjectGNN(in_channels=in_channels, hidden_channels=hidden_channels, dropout=dropout).to(DEVICE)


model = build_model()
print(model)

## Training Loop

Weighted binary cross-entropy, Adam optimizer, early stopping on weighted
validation AUC - same convention as `DNN.ipynb`'s `train_model`. Every epoch
also runs an eval-mode pass over the training graphs (dropout off, plain
`|w_phys|` weights via `w_eval`) so the logged `train_auc_eval`/`val_auc`
pair is directly comparable (the dropout-on `train_auc` used for the
gradient step is not).

In [ ]:
N_EPOCHS = 3
PATIENCE = 3
LEARNING_RATE = 1e-3


def run_epoch(model, loader, criterion, optimizer, train, weight_attr="w"):

    """One pass over `loader`'s batches, optionally taking a gradient step."""

    model.train(train)
    total_loss, total_weight = 0.0, 0.0
    all_labels, all_probs, all_weights = [], [], []

    with torch.set_grad_enabled(train):
        for batch in loader:
            batch = batch.to(DEVICE)
            w = getattr(batch, weight_attr)

            logits = model(batch.x, batch.edge_index, batch.batch)
            loss = (criterion(logits, batch.y) * w).sum() / w.sum()

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            bw = w.sum().item()
            total_loss += loss.item() * bw
            total_weight += bw
            all_labels.append(batch.y.detach().cpu().numpy())
            all_probs.append(torch.sigmoid(logits).detach().cpu().numpy())
            all_weights.append(w.detach().cpu().numpy())

    labels = np.concatenate(all_labels)
    probs = np.concatenate(all_probs)
    weights = np.concatenate(all_weights)
    auc = roc_auc_score(labels, probs, sample_weight=weights)

    return total_loss / total_weight, auc


def train_model(hidden_channels=DEFAULT_HIDDEN_CHANNELS, dropout=DEFAULT_DROPOUT, lr=LEARNING_RATE,
                 n_epochs=N_EPOCHS, patience=PATIENCE, verbose=True,
                 train_loader_fit_data=None, train_loader_eval_data=None, val_loader_data=None):

    """
    Build a fresh ObjectGNN and train it with early stopping on weighted
    validation AUC. Returns (model, history, best_val_auc, best_train_auc,
    best_train_auc_eval) - `model` already has the best-epoch weights
    loaded, and the two train_auc numbers are read off that SAME epoch.
    """

    train_loader_fit_data = train_loader_fit if train_loader_fit_data is None else train_loader_fit_data
    train_loader_eval_data = train_loader_eval if train_loader_eval_data is None else train_loader_eval_data
    val_loader_data = val_loader if val_loader_data is None else val_loader_data

    trial_model = build_model(hidden_channels=hidden_channels, dropout=dropout)
    criterion = nn.BCEWithLogitsLoss(reduction="none")
    optimizer = torch.optim.Adam(trial_model.parameters(), lr=lr)

    history = {"train_loss": [], "val_loss": [], "train_auc": [], "val_auc": [],
               "train_loss_eval": [], "train_auc_eval": []}
    best_val_auc, best_train_auc, best_train_auc_eval = -np.inf, None, None
    best_state, epochs_no_improve = None, 0

    for epoch in range(1, n_epochs + 1):

        train_loss, train_auc = run_epoch(trial_model, train_loader_fit_data, criterion, optimizer, train=True, weight_attr="w")
        train_loss_eval, train_auc_eval = run_epoch(trial_model, train_loader_eval_data, criterion, optimizer, train=False, weight_attr="w_eval")
        val_loss, val_auc = run_epoch(trial_model, val_loader_data, criterion, optimizer, train=False, weight_attr="w")

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_auc"].append(train_auc)
        history["val_auc"].append(val_auc)
        history["train_loss_eval"].append(train_loss_eval)
        history["train_auc_eval"].append(train_auc_eval)

        if verbose:
            print(f"Epoch {epoch:3d} | train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
                  f"| train_auc={train_auc:.4f} train_auc_eval={train_auc_eval:.4f} val_auc={val_auc:.4f}")

        if val_auc > best_val_auc:
            best_val_auc, best_train_auc, best_train_auc_eval = val_auc, train_auc, train_auc_eval
            best_state, epochs_no_improve = {k: v.clone() for k, v in trial_model.state_dict().items()}, 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch} (best val_auc={best_val_auc:.4f})")
                break

    trial_model.load_state_dict(best_state)
    return trial_model, history, best_val_auc, best_train_auc, best_train_auc_eval

In [ ]:
model, history, best_val_auc, best_train_auc, best_train_auc_eval = train_model()

print(f"\nBest val_auc = {best_val_auc:.4f} | train_auc (dropout on) = {best_train_auc:.4f} "
      f"| train_auc_eval (dropout off, comparable) = {best_train_auc_eval:.4f}")

## Evaluation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(history["train_loss"], label="train (dropout on)", color="tab:blue", alpha=0.5)
axes[0].plot(history["train_loss_eval"], label="train (dropout off, comparable)", color="tab:blue")
axes[0].plot(history["val_loss"], label="val", color="tab:orange")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("weighted BCE loss")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(history["train_auc"], label="train (dropout on)", color="tab:blue", alpha=0.5)
axes[1].plot(history["train_auc_eval"], label="train (dropout off, comparable)", color="tab:blue")
axes[1].plot(history["val_auc"], label="val", color="tab:orange")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("weighted AUC")
axes[1].set_title("AUC")
axes[1].legend()

plt.tight_layout()
plt.show()

model.eval()
all_y, all_p, all_w = [], [], []
with torch.no_grad():
    for batch in val_loader:
        batch = batch.to(DEVICE)
        probs = torch.sigmoid(model(batch.x, batch.edge_index, batch.batch))
        all_y.append(batch.y.cpu().numpy())
        all_p.append(probs.cpu().numpy())
        all_w.append(batch.w.cpu().numpy())

y_val_scores = np.concatenate(all_p)
y_val = np.concatenate(all_y)
w_val = np.concatenate(all_w)

auc_val = roc_auc_score(y_val, y_val_scores, sample_weight=w_val)
print(f"Final weighted val AUC = {auc_val:.4f}")

fpr, tpr, _ = roc_curve(y_val, y_val_scores, sample_weight=w_val)
plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f"GNN (AUC = {auc_val:.4f})")
plt.plot([0, 1], [0, 1], "k--", label="Random")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curve (validation)")
plt.legend()
plt.tight_layout()
plt.show()

## Physics Figure of Merit & Held-Out Test Evaluation

Weighted AUC is a global ranking metric; for HH what matters is significance
in the high-score region - same max-Asimov-significance scan as
`DNN.ipynb`. The score cut is selected by scanning **VAL** only, then applied
**frozen** to the never-before-seen **TEST** partition, scored exactly once.

In [ ]:
def significance_scan(y_true, scores, w_phys, n_thr=200, min_bkg=1.0):

    """
    Max Asimov significance over score cuts. Uses SIGNED w_phys (expected
    yields, not |w_phys|). `min_bkg` guards the sparse high-score tail
    where Z is unstable when B is tiny or negative. Treat Z as a RELATIVE
    metric for ranking models on identical rows, not an absolute discovery
    number (it depends on sample normalization/luminosity).
    """

    thr = np.quantile(scores, np.linspace(0, 1, n_thr))
    best_z, best_t = 0.0, None

    for t in thr:
        sel = scores >= t
        S = w_phys[sel & (y_true == 1)].sum()
        B = w_phys[sel & (y_true == 0)].sum()
        if S <= 0 or B < min_bkg:
            continue
        z = np.sqrt(2 * ((S + B) * np.log(1 + S / B) - S))
        if z > best_z:
            best_z, best_t = z, t

    return best_z, best_t


w_val_signed = val_df["w_phys"].to_numpy()
z_val, thr_val = significance_scan(y_val, y_val_scores, w_val_signed)

print(f"Weighted val AUC = {auc_val:.4f}")
print(f"Max Asimov Z (val) = {z_val:.3f} at score cut = {thr_val:.4f}")

# ---- HELD-OUT TEST EVALUATION - scored EXACTLY ONCE ------------------------
# test_dataset/test_loader have not participated in anything above (not the
# imputation/scaler fit, not training, not early stopping, not the
# significance-scan cut selection just above). The cut `thr_val` was chosen
# on VAL only; it is applied here FROZEN, NOT re-scanned on test.

model.eval()
all_y, all_p, all_w = [], [], []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(DEVICE)
        probs = torch.sigmoid(model(batch.x, batch.edge_index, batch.batch))
        all_y.append(batch.y.cpu().numpy())
        all_p.append(probs.cpu().numpy())
        all_w.append(batch.w.cpu().numpy())

test_scores = np.concatenate(all_p)
y_test = np.concatenate(all_y)
w_test_abs = np.concatenate(all_w)
w_test_signed = test_df["w_phys"].to_numpy()

auc_test = roc_auc_score(y_test, test_scores, sample_weight=w_test_abs)

S_test = w_test_signed[(test_scores >= thr_val) & (y_test == 1)].sum()
B_test = w_test_signed[(test_scores >= thr_val) & (y_test == 0)].sum()
z_test = (np.sqrt(2 * ((S_test + B_test) * np.log(1 + S_test / B_test) - S_test))
          if (S_test > 0 and B_test > 0) else np.nan)

print(f"\nWeighted AUC: val = {auc_val:.4f}  |  test (held-out, scored once) = {auc_test:.4f}")
print(f"At the VAL-selected score cut = {thr_val:.4f} (frozen, NOT re-scanned on test):")
print(f"  test S = {S_test:.2f} | test B = {B_test:.2f} | test Z = {z_test:.3f}  (val Z was {z_val:.3f})")

## Sanity Checks & Summary

In [ ]:
# ---- Determinism check (same acceptance criterion as DNN.ipynb): re-seeding
# immediately before each of two short training runs should give an EXACT
# match, proving set_seed()/use_deterministic_algorithms(True) pin every
# source of randomness. Cheap config (small model, 5 epochs) purely to keep
# this check fast - not meant to reflect final model quality.

set_seed(RANDOM_STATE)
_, _, det_check_a, _, _ = train_model(hidden_channels=16, n_epochs=2, patience=2, verbose=False)

set_seed(RANDOM_STATE)
_, _, det_check_b, _, _ = train_model(hidden_channels=16, n_epochs=2, patience=2, verbose=False)
assert det_check_a == det_check_b, f"Determinism check FAILED: {det_check_a} != {det_check_b}"

print(f"Determinism check passed: two re-seeded runs give identical val_auc = {det_check_a:.6f}")

# ---- Final summary ----------------------------------------------------------

summary = pd.DataFrame([{
    "model": f"GNN (object nodes, {N_NODES} nodes/event)",
    "val_auc": auc_val,
    "max_asimov_Z (val)": z_val,
    "test_auc (held-out, scored once)": auc_test,
    "test_Z_at_val_cut (held-out)": z_test,
}])

print("\nFinal summary (GNN, 1l2tau Run 2):")
print(summary.to_string(index=False))
summary